# Launch Branch Workloads From JupyterHub

This workbook installs the service-workloads-jupyterhub-auth branch into the notebook user environment, then uses the JupyterHub API token to declare, validate, run, and proxy Goblin King workloads before the branch is merged.

In [ ]:
import importlib
import json
import os
import site
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

BRANCH_PACKAGE = "git+https://github.com/tashabits/goblin-king.git@service-workloads-jupyterhub-auth"
package = os.environ.get("GOBLIN_KING_BRANCH_NOTEBOOK_PACKAGE", BRANCH_PACKAGE)
print(f"Installing notebook helper from {package}")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--quiet",
    "--user",
    "--force-reinstall",
    "--no-deps",
    package,
])
print("Notebook helper install complete")
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "goblin_king" or module_name.startswith("goblin_king."):
        del sys.modules[module_name]

import goblin_king  # noqa: E402, I001
from goblin_king.notebooks import GoblinKingNotebookClient  # noqa: E402, I001

if "JUPYTERHUB_API_TOKEN" not in os.environ:
    raise RuntimeError("JUPYTERHUB_API_TOKEN is required; run this inside a JupyterHub user server")

GOBLIN_KING_API_URL = os.environ.get(
    "GOBLIN_KING_API_URL",
    "http://goblin-king-api.default.svc.cluster.local:8000",
).rstrip("/")
JUPYTERHUB_TOKEN = os.environ["JUPYTERHUB_API_TOKEN"]
client = GoblinKingNotebookClient(
    api_url=GOBLIN_KING_API_URL,
    token=JUPYTERHUB_TOKEN,
    request_timeout_seconds=120,
)

print(f"Loaded goblin_king from {Path(goblin_king.__file__).resolve()}")
print(f"Using Goblin King API at {GOBLIN_KING_API_URL}")

def goblin_request(path, method="GET", payload=None):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {"Authorization": f"Bearer {JUPYTERHUB_TOKEN}", "Accept": "application/json"}
    if body is not None:
        headers["Content-Type"] = "application/json"
    request = urllib.request.Request(
        GOBLIN_KING_API_URL + path,
        data=body,
        headers=headers,
        method=method,
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            text = response.read().decode("utf-8")
            return json.loads(text) if text else {"status": response.status}
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Goblin King API returned {error.code}: {detail}") from error

def require_api_paths(*paths):
    spec = goblin_request("/openapi.json")
    available = set(spec.get("paths", {}))
    missing = [path for path in paths if path not in available]
    if missing:
        raise RuntimeError(
            "Goblin King API at "
            f"{GOBLIN_KING_API_URL} does not expose required notebook routes: {missing}. "
            "The notebook helper is installed, but the API deployment is older or "
            "GOBLIN_KING_API_URL points at the wrong service. Redeploy Goblin King from "
            "the service-workloads-jupyterhub-auth branch, or set GOBLIN_KING_API_URL "
            "to the matching API service."
        )
    print(f"Goblin King API exposes notebook routes: {', '.join(paths)}")

require_api_paths("/notebooks/goblins", "/notebooks/services")

GOBLIN_KING_API_URL

In [ ]:
def branch_workbook_hello(payload):
    name = payload.get("name", "Branch Workbook")
    return {
        "message": f"Hello {name}",
        "source": "branch-pinned-workbook",
        "name_length": len(name),
    }

workbook_goblin = client.declare(
    branch_workbook_hello,
    kind="notebook.branch-workbook-hello",
    display_name="Branch Workbook Hello",
    timeout_seconds=30,
)
workbook_goblin.record

In [ ]:
validation = workbook_goblin.validate({"name": "Validation"})
validation["validation"]

In [ ]:
run = workbook_goblin.run(
    {"name": "JupyterHub"},
    progress=True,
    progress_interval_seconds=2,
)
run["run"]["result"]

In [ ]:
goblins = goblin_request("/goblins")
[item["kind"] for item in goblins if item["kind"].startswith("notebook.")]

In [ ]:
SERVICE_SOURCE = """
from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def hello():
    return {
        "message": "Hello World",
        "source": "notebook-defined-asgi-service",
    }
""".strip()

try:
    client.stop_service("notebook.workbook-long-hello")
except RuntimeError as error:
    if "404" not in str(error):
        raise

long_service = client.declare_service(
    source=SERVICE_SOURCE,
    kind="notebook.workbook-long-hello",
    display_name="Workbook Long Hello Service",
    app_name="app",
    requirements=["fastapi>=0.115,<1"],
    probe_path="/hello",
    project_id="default",
)
long_service.record


In [ ]:
service_validation = long_service.validate(timeout_seconds=180)
service_validation["runtime"]


In [ ]:
service_start = long_service.start(timeout_seconds=180, progress=True)
{
    "service_id": service_start["service"]["id"],
    "runtime": service_start["runtime"],
    "probe": service_start["probe"]["response"].get("json"),
}


In [ ]:
probe = long_service.probe()
proxied = long_service.proxy("/hello")
{
    "probe_json": probe["response"].get("json"),
    "proxied_json": proxied,
}


In [ ]:
stopped = long_service.stop()
stopped["notebook_service"]["runtime_status"]
